# 06 – Single-Agent PPO for Credit Decisioning

**Project:** Multi-Agent DRL + CNN Alternative Data

This notebook trains a Proximal Policy Optimization (PPO) agent on the fused features. It provides a stronger single-agent baseline than the earlier DQN.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("..")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
X = np.load(DATA_PROCESSED / "X_fused.npy")
y = np.load(DATA_PROCESSED / "y.npy")
thin = np.load(DATA_PROCESSED / "thin.npy")

STATE_DIM = X.shape[1]
N_ACTIONS = 2
print(f"State dim: {STATE_DIM}, Samples: {len(y)}")

In [ ]:
class CreditEnv:
    def __init__(self, X, y, thin):
        self.X = X
        self.y = y
        self.thin = thin
        self.n = len(y)
        self.idx = 0

    def reset(self):
        self.idx = np.random.randint(0, self.n)
        return self.X[self.idx]

    def step(self, action):
        true_label = self.y[self.idx]
        is_thin = self.thin[self.idx]

        if action == 0:  # Approve
            if true_label == 0:
                reward = 1.0 + (0.5 if is_thin else 0.0)
            else:
                reward = -5.0
        else:  # Reject
            if true_label == 1:
                reward = 5.0
            else:
                reward = -1.0

        done = True
        self.idx = (self.idx + 1) % self.n
        return self.X[self.idx], reward, done, {}

env = CreditEnv(X, y, thin)
print("Environment ready.")

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        self.actor = nn.Linear(64, n_actions)
        self.critic = nn.Linear(64, 1)

    def forward(self, x):
        features = self.shared(x)
        return self.actor(features), self.critic(features)

model = ActorCritic(STATE_DIM, N_ACTIONS).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
print(model)

In [ ]:
EPISODES = 800
GAMMA = 0.95
CLIP_EPS = 0.2
EPOCHS_PER_UPDATE = 4

rewards_history = []

for ep in range(1, EPISODES + 1):
    states, actions, rewards, log_probs, values, dones = [], [], [], [], [], []
    
    state = env.reset()
    total_r = 0
    
    for t in range(32):
        state_t = torch.tensor(state, dtype=torch.float32, device=device)
        logits, value = model(state_t)
        dist = Categorical(logits=logits)
        action = dist.sample()
        
        next_state, reward, done, _ = env.step(action.item())
        
        states.append(state)
        actions.append(action.item())
        rewards.append(reward)
        log_probs.append(dist.log_prob(action))
        values.append(value)
        dones.append(done)
        
        state = next_state
        total_r += reward
        if done:
            break
    
    returns = []
    R = 0
    for r, d in zip(reversed(rewards), reversed(dones)):
        R = r + GAMMA * R * (1 - d)
        returns.insert(0, R)
    
    returns = torch.tensor(returns, dtype=torch.float32, device=device)
    advantages = returns - torch.stack(values).detach().squeeze()
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    states_t = torch.tensor(np.array(states), dtype=torch.float32, device=device)
    actions_t = torch.tensor(actions, dtype=torch.int64, device=device)
    old_log_probs = torch.stack(log_probs).detach()
    
    for _ in range(EPOCHS_PER_UPDATE):
        logits, values_pred = model(states_t)
        dist = Categorical(logits=logits)
        new_log_probs = dist.log_prob(actions_t)
        entropy = dist.entropy().mean()
        
        ratio = (new_log_probs - old_log_probs).exp()
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()
        critic_loss = nn.MSELoss()(values_pred.squeeze(), returns)
        
        loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    rewards_history.append(total_r)
    
    if ep % 100 == 0:
        avg_r = np.mean(rewards_history[-100:])
        print(f"Episode {ep:4d} | Avg Reward: {avg_r:7.2f}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(pd.Series(rewards_history).rolling(50).mean())
plt.title("Single-Agent PPO – Smoothed Episode Reward")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.tight_layout()
plt.savefig(RESULTS / "ppo_reward.png", dpi=120)
plt.show()

torch.save(model.state_dict(), RESULTS / "single_agent_ppo.pt")
print("Saved → results/single_agent_ppo.pt")